# YouTube Virality Predictor: Building a Multimodal AI Pipeline

This notebook documents the end-to-end development of the **YouTube Virality Predictor**, an AI system that predicts video view counts based on thumbnails, titles, and channel metadata.

### Project Goal
To create a model capable of understanding what make a video "go viral" by analyzing the visual, textual, and social context of a upload.

## 1. Environment Setup

The project relies on `PyTorch` for the neural network, `torchvision` for image processing, and `yt-dlp` for high-speed metadata scraping.

In [ ]:
!pip install yt-dlp torch torchvision pandas pillow matplotlib

## 2. Data Acquisition Strategy

We collected a dataset of over **4,300+ records**. 

### Scraping Logic
- **Tool:** `yt-dlp` with `skip_download=True` and `write_info_json=True`.
- **Niche Targeting:** Specifically prioritized **Linux Tech Content** and **Video Game Comedy Skits** to diversify the dataset beyond standard trending fare.
- **Rate Limiting:** Implemented a 5-10 second sleep between queries to avoid IP bans from YouTube's anti-bot systems.

## 3. Data Preprocessing

Raw JSON data was compiled into a single `dataset.csv`. 

### Key Transformations:
1. **Log-Scaling Targets:** Instead of predicting raw view counts (which vary from 0 to 100M+), we predicted `log1p(view_count)`. This stabilizes the loss function and allows the model to treat the difference between 10k and 100k views as being as significant as 1M to 10M.
2. **Z-Score Normalization:** Subscriber counts, durations, and title lengths were normalized using mean/std for faster convergence.
3. **Categorical Encoding:** Channel names and video categories were mapped to unique embedding indices.

## 4. The `ViralityNet` Architecture

The model is a **Multimodal Fusion Network**.

### Branch A: Vision (The Thumbnail)
- **Backbone:** `MobileNetV3-Small` (Pretrained on ImageNet).
- **Fine-tuning:** The last two blocks of the backbone were **unfrozen** to allow the model to adapt to YouTube's specific vibrancy and text-over-image styles.

### Branch B: Text (The Title)
- **Logic:** A **Bidirectional GRU** (Gated Recurrent Unit).
- **Benefit:** Captures context from both the start and end of a title simultaneously.

### Branch C: Metadata (The Context)
- **Follower Embedding:** A 16-dimension embedding for the specific channel posting the video.
- **Numerical Context:** Ingests normalized subscriber counts, duration, and upload timing.

In [ ]:
# Simplified architectural look
import torch.nn as nn

class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Vision Path
        self.vision_proj = nn.Linear(576, 256)
        # Text Path
        self.rnn = nn.GRU(64, 64, bidirectional=True)
        # Meta Path
        self.meta_fc = nn.Linear(5 + 16 + 16, 32)
        # Fusion
        self.head = nn.Linear(256 + 128 + 32, 1) # Final prediction

## 5. Training Loop

- **Loss Function:** `MSELoss` (Mean Squared Error).
- **Optimizer:** `AdamW` with a weight decay of `1e-4` to prevent overfitting.
- **Scheduler:** `CosineAnnealingLR` to smoothly decay the learning rate over 25 epochs.
- **Regularization:** Heavy `Dropout(0.4)` used in the fusion head to ensure the model doesn't just memorize specific creator names.

## 6. Results & Deployment

The best model reached a validation MSE loss of **~5.16**. 

### Final Packaging
- **Frontend:** Vanilla HTML/JS with a futuristic glassmorphism theme.
- **Backend:** Flask API server.
- **Release:** Bundled into a standalone Windows `.exe` via **PyInstaller** for easy portability without needing Python installed.

--- 
*Developed as part of the AI Training Series 2026*